# 03. Preliminary model with logistic regression (baseline, Figure S2)

Same data, prediction tasks (ballot × year), 5-fold CV × 20 repetitions and survey weights as
`02_preliminary.ipynb`, with an L2-regularized logistic regression instead of XGBoost.

- Data: the list-wise-deleted preliminary data, so no imputation is needed.
- Encoding of categorical variables (two versions):
  - `Logit_OneHot`: variables with ≤20 distinct values are one-hot encoded, the rest are standardized;
  - `Logit_Numeric`: all variables are treated as numeric and standardized.
- Model: `LogisticRegressionCV` (L2, regularization strength chosen by 3-fold CV within the training fold).
- SHAP: `LinearExplainer` on the transformed matrix; one-hot columns are summed back to the original variable.

**Input**: `data/cleaned_data_for_preliminary_analysis.pkl` (from `01_make_data.ipynb`)

**Output**: `outputs/1_1_preliminary_logit/1_Logit_{OneHot,Numeric}_partyid_1_*` (Figure S2)

## Model training and evaluation (one run)

In [ ]:
import pandas as pd
import numpy as np
import shap
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegressionCV

# Max number of distinct values for a variable to be treated as categorical in 'onehot' encoding
MAX_CAT_UNIQUE = 20


def build_logit_pipeline(con_cols, nom_cols):
    """
    Continuous: standardized. Categorical: one-hot (categories with <10 cases are pooled; categories unseen in the training fold map to the pool or to zeros).
    Model: L2 logistic regression; regularization strength C chosen by 3-fold CV (ROC AUC) within the training fold.
    (The preliminary data are already list-wise deleted, so no imputation is needed.)
    """
    preprocessor = ColumnTransformer([
        ('con', StandardScaler(), con_cols),
        ('nom', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=10,
                              sparse_output=False), nom_cols)
    ], verbose_feature_names_out=False)
    return Pipeline([
        ('preprocess', preprocessor),
        ('logit', LogisticRegressionCV(Cs=10, cv=3, penalty='l2', scoring='roc_auc', max_iter=5000))
    ])


def onehot_to_feature_matrix(transformed_names, feature_names, nom_cols):
    """
    0/1 matrix (n_transformed x n_features) that sums transformed columns (1 per continuous variable, several one-hot columns per categorical variable) back to the original features.
    SHAP values are additive, so summing the SHAP values of the one-hot columns gives the SHAP value of the original variable.
    """
    feature_idx = {f: i for i, f in enumerate(feature_names)}
    nom_set = set(nom_cols)
    nom_sorted = sorted(nom_cols, key=len, reverse=True)  # longest name first when names overlap (e.g. reltrad / reltrad16)
    mapping = np.zeros((len(transformed_names), len(feature_names)))
    for j, name in enumerate(transformed_names):
        if name in feature_idx and name not in nom_set:
            owner = name  # continuous variables keep their names
        else:
            owner = next((f for f in nom_sorted if name.startswith(f + '_')), None)
        # A column that belongs to no feature would make the sum wrong, so stop.
        if owner is None:
            raise ValueError(f"Transformed column '{name}' cannot be mapped to an original feature.")
        mapping[j, feature_idx[owner]] = 1
    # A feature without any transformed column would misalign the SHAP dimensions, so stop.
    empty = [f for f, i in feature_idx.items() if mapping[:, i].sum() == 0]
    if empty:
        raise ValueError(f"Features without any transformed column: {empty}")
    return mapping

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def run_single_ml_iteration_weighted(
    df_dict,          # input data: {(ballot, model): {year: df}}
    target_col='partyid_1',
    random_seed=42,
    run_number=1,
    use_weights=True,
    encoding='onehot'  # 'onehot': <= MAX_CAT_UNIQUE distinct values = categorical (one-hot) / 'numeric': all continuous
):
    """
    For each (ballot, model) and year, train and evaluate the model with 5-fold CV
    and return the SHAP importance averaged over folds.
    """
    scores_list = []
    importances_list = []
    dimensions_list = []
    feature_usage_list = []

    for (ballot, model_name), year_dict in df_dict.items():

        # This notebook only uses logistic regression ('Logit'). Any other model_name means
        # the upstream data structure is broken, so stop immediately.
        if model_name != 'Logit':
            raise ValueError(f"Unknown model_name: '{model_name}' (ballot={ballot})")

        for year, df_original in year_dict.items():
            dataset_identifier = f"({ballot}, {model_name}) - {year}"

            df = df_original.copy()
            if not df.index.is_unique:
                df.reset_index(drop=True, inplace=True)

            # Upstream steps should only create non-empty data that contain target_col.
            # A violation indicates an upstream bug, so stop.
            if df.empty:
                raise ValueError(f"{dataset_identifier} - empty data frame.")
            if target_col not in df.columns:
                raise ValueError(f"{dataset_identifier} - target column '{target_col}' not found.")

            target_series = df[target_col]
            if isinstance(target_series, pd.DataFrame):
                target_series = target_series.iloc[:, 0]
            df_filtered = df[target_series.notna() & (target_series != 0)].copy()

            # No cases after filtering means the year/ballot data must be checked.
            if df_filtered.empty:
                raise ValueError(f"{dataset_identifier} - no cases left after filtering '{target_col}'.")
            df_filtered.reset_index(drop=True, inplace=True)

            y_original = df_filtered[target_col]
            if isinstance(y_original, pd.DataFrame):
                y_original = y_original.iloc[:, 0]

            # GSS weights: wtssnrps from 2004 on, wtssps before; equal weights (1) if neither exists
            weights_original = pd.Series(np.ones(len(df_filtered)), index=df_filtered.index)
            if use_weights:
                if 'wtssnrps' in df_filtered.columns and pd.to_numeric(year, errors='coerce') >= 2004:
                    weights_original = df_filtered['wtssnrps']
                elif 'wtssps' in df_filtered.columns:
                    weights_original = df_filtered['wtssps']
                weights_original = weights_original.fillna(1)

            # 5-fold CV needs at least 20 cases and 2 classes.
            if len(df_filtered) < 20 or y_original.nunique() < 2:
                raise ValueError(
                    f"{dataset_identifier} - too few cases ({len(df_filtered)}) or "
                    f"too few classes of '{target_col}' ({y_original.nunique()})."
                )

            # Variables excluded from the party identity model:
            # potential_targets: direct/indirect measures of party identity, ideology, or vote choice (avoid leakage)
            potential_targets = [
                'partyid_1', 'partyid_2', 'partyid_3', 'partyid',
                'partyid1', 'partyid2', 'leftrght', 'polviews_c',
                'polintrst', 'ratechall124', 'ratechall1ts',
                'ratechall224', 'ratechall2ts', 'ratechall324', 'ratechall3ts', 'ratedemp',
                'rateincumb24', 'rateincumbts', 'ratemaga', 'raterepp',
                'whovote24', 'whovote24a', 'whovotets',
                'pres68', 'if68who', 'pres72', 'if72who', 'pres76', 'if76who', 'pres80', 'if80who',
                'pres84', 'if84who', 'pres88', 'if88who', 'pres92', 'if92who', 'pres96', 'if96who',
                'pres00', 'if00who', 'pres04', 'if04who', 'pres08', 'if08who', 'pres12', 'if12who',
                'pres16', 'if16who', 'polviews', 'polviewy', 'polviewx', 'pres20', 'if20who',
                'leftrght1', 'prespop', 'libtemp', 'contemp', 'acqdems', 'acqreps', 'acqlib', 'acqcon',
            ]
            # not_needed: survey metadata, weights, duplicated variables, etc.
            not_needed = [
                'hlthstrt', 'huadd', 'huaddwhy', 'dwellpre', 'kidsinhh', 'respond',
                'incuspop', 'neisafe', 'rlooks', 'rgroomed', 'rweight', 'rhlthend', 'huclean', 'coop',
                'comprend', 'consent', 'adminconsent',
                'whoelse1', 'whoelse2', 'whoelse3', 'whoelse4', 'whoelse5', 'whoelse6',
                'lngthinv', 'intage', 'intage_0418', 'intethn', 'intsex', 'intyrs', 'intyrs_0418', 'inthisp',
                'intrace1', 'intrace2', 'intrace3',
                'ratetone', 'mode', 'modesequence', 'feeused', 'feelevel', 'totalincentive',
                'spaneng', 'spanint', 'spanself', 'formwt', 'oversamp',
                'wtss', 'wtssall', 'wtssnr', 'wtsps', 'wtssps',
                'wtssnrps', 'ballotformwt', 'ballotformwtnr', 'wtsps_as', 'wtssnrps_as',
                'id', 'hefinfo', 'hefinfo1', 'respnum', 'respnumh',
                'year', 'dateintv', 'ballot', 'version', 'form', 'issp', 'phone',
                'sample', 'sampcode', 'phase', 'subsamprate', 'batch', 'kish',
                'vpsu', 'vstrat', 'fileversion', 'final_weight', 'intid', 'genevig1', 'genevig2',
                # mostimp: unclear and too many response categories
                'mostimp1', 'mostimp2', 'mostimp3', 'mostimp4',
                'prob1', 'prob2', 'prob3', 'prob4', 'extra1', 'extra2',
                # srcbelt, size: place-size variables that duplicate xnorcsiz
                'srcbelt', 'size',
                # *_7222: duplicates of variables that already cover all years
                'babies_7222', 'earnrs_7222', 'hompop_7222', 'preteen_7222',
                'reg16_7222', 'region_7222', 'sibs_7222', 'teens_7222',
                # Interviewer-related variables
                'saqsex', 'saqissp', 'easyget', 'scimode', 'sciintro', 'rundrstd',
                'robject', 'intrwght', 'saqgene', 'coop2',
                # hhtype: duplicates hhtype1
                'hhtype',
            ]

            features_to_drop = (
                [col for col in potential_targets if col in df_filtered.columns] +
                [col for col in not_needed if col in df_filtered.columns]
            )
            X = df_filtered.drop(columns=features_to_drop, errors='ignore')

            # List-wise deletion: drop cases with a missing value in any
            # of the variables used in the model.
            X = X.dropna()
            y_original = y_original.loc[X.index]
            weights_original = weights_original.loc[X.index]

            # Re-check the number of cases/classes after list-wise deletion.
            # Skip with a warning when data are too thin (e.g. ballot rotation).
            if len(y_original) < 20 or y_original.nunique() < 2:
                print(
                    f"[SKIP] {dataset_identifier} - too few cases ({len(y_original)}) or "
                    f"classes ({y_original.nunique()}) after list-wise deletion."
                )
                continue

            feature_names = X.columns.tolist()

            # No features left means the data were built incorrectly.
            if not feature_names:
                raise ValueError(f"{dataset_identifier} - no features (all columns excluded).")

            feature_usage_list.append({
                'run': run_number, 'ballot': ballot, 'year': year,
                'model': model_name, 'features_used': ','.join(feature_names)
            })

            y_encoded = LabelEncoder().fit_transform(y_original)
            n_classes = len(np.unique(y_encoded))

            dimensions_list.append({
                'run': run_number, 'ballot': ballot, 'year': year,
                'model': model_name, 'n_rows': len(X), 'n_cols': len(feature_names)
            })

            # Categorical vs. continuous (preliminary variables carry no type information, so a rule is used)
            if encoding == 'onehot':
                nom_cols = [c for c in feature_names if X[c].nunique() <= MAX_CAT_UNIQUE]
            elif encoding == 'numeric':
                nom_cols = []
            else:
                raise ValueError(f"Unknown encoding: '{encoding}'")
            con_cols = [c for c in feature_names if c not in nom_cols]
            # X for the logit model: categorical variables as string categories (OOF results and importance keep the original feature names)
            X_model = X.copy()
            for col in nom_cols:
                X_model[col] = X[col].astype(str)

            kfold = KFold(n_splits=5, shuffle=True, random_state=random_seed)
            fold_scores = {'accuracy': [], 'f1_macro': [], 'roc_auc_ovr': []}
            fold_importances = []

            for fold_i, (train_idx_rel, test_idx_rel) in enumerate(kfold.split(X, y_encoded)):
                train_abs = X.index[train_idx_rel]
                test_abs  = X.index[test_idx_rel]
                X_train, X_test = X_model.loc[train_abs], X_model.loc[test_abs]
                y_train, y_test = y_encoded[train_idx_rel], y_encoded[test_idx_rel]
                w_train, w_test = weights_original.loc[train_abs], weights_original.loc[test_abs]

                # Preprocessing (standardization, one-hot) is fit inside the pipeline on the training fold only, so no test-fold information leaks.
                model = build_logit_pipeline(con_cols, nom_cols)
                fit_params = {'logit__sample_weight': w_train} if use_weights else {}
                model.fit(X_train, y_train, **fit_params)

                y_pred  = model.predict(X_test)
                y_proba = model.predict_proba(X_test)
                fold_scores['accuracy'].append(accuracy_score(y_test, y_pred, sample_weight=w_test))
                fold_scores['f1_macro'].append(f1_score(y_test, y_pred, average='macro', sample_weight=w_test))
                if n_classes > 2:
                    fold_scores['roc_auc_ovr'].append(
                        roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro', sample_weight=w_test))
                else:
                    fold_scores['roc_auc_ovr'].append(
                        roc_auc_score(y_test, y_proba[:, 1], average='macro', sample_weight=w_test))

                if X_test.empty:
                    fold_importances.append(np.array([np.nan] * len(feature_names)))
                    continue

                # SHAP from LinearExplainer on the transformed matrix Z (background = training fold),
                # then the one-hot columns are summed back to the original features.
                preprocess, logit = model.named_steps['preprocess'], model.named_steps['logit']
                Z_train, Z_test = preprocess.transform(X_train), preprocess.transform(X_test)
                explainer = shap.LinearExplainer(logit, Z_train)
                shap_z = explainer.shap_values(Z_test)
                # SHAP sum + expected value must equal the model's log-odds output; stop otherwise.
                additivity_err = np.abs(shap_z.sum(axis=1) + explainer.expected_value - logit.decision_function(Z_test)).max()
                if additivity_err > 1e-6:
                    raise ValueError(f"{dataset_identifier} fold {fold_i} - SHAP additivity violated (max error {additivity_err:.2e})")
                shap_out = shap_z @ onehot_to_feature_matrix(preprocess.get_feature_names_out().tolist(), feature_names, nom_cols)

                if isinstance(shap_out, list):
                    # A list of SHAP arrays must have one element per class.
                    if len(shap_out) != n_classes:
                        raise ValueError(
                            f"{dataset_identifier} fold {fold_i} - length of SHAP list ({len(shap_out)}) "
                            f"does not match the number of classes ({n_classes})."
                        )
                    shap_arr = shap_out[1]
                elif isinstance(shap_out, np.ndarray):
                    shap_arr = shap_out
                else:
                    raise ValueError(
                        f"{dataset_identifier} fold {fold_i} - unexpected SHAP output type: {type(shap_out)}"
                    )

                mean_abs = np.abs(shap_arr).mean(axis=0)
                # SHAP dimensions must match the number of features.
                if len(mean_abs) != len(feature_names):
                    raise ValueError(
                        f"{dataset_identifier} fold {fold_i} - SHAP dimension ({len(mean_abs)}) "
                        f"does not match the number of features ({len(feature_names)})."
                    )
                fold_importances.append(mean_abs)

            # Collect fold results
            for metric, values in fold_scores.items():
                if values:
                    scores_list.append({
                        'ballot': ballot, 'model': model_name, 'year': year, 'run': run_number,
                        'metric': metric, 'avg_score': np.mean(values), 'std_score': np.std(values)
                    })

            if fold_importances:
                avg_imp = np.nanmean(np.array(fold_importances), axis=0)
                for feat, imp in zip(feature_names, avg_imp):
                    if not np.isnan(imp):
                        importances_list.append({
                            'ballot': ballot, 'model': model_name, 'year': year, 'run': run_number,
                            'feature': feat, 'importance': float(imp)
                        })

    return (
        pd.DataFrame(scores_list), pd.DataFrame(importances_list),
        pd.DataFrame(dimensions_list), pd.DataFrame(feature_usage_list)
    )

## Repeated runs

In [ ]:
import pandas as pd
import numpy as np

def run_ml_multiple_times_weighted(
    data_for_analysis_dict,  # input data: {(ballot, model): {year: df}}
    target_col='partyid_1',
    n_runs=3,
    use_weights=True,
    encoding='onehot'
):
    """
    Run run_single_ml_iteration_weighted n_runs times and combine the results.
    Each run uses its own random_seed (= run index), so results are reproducible.
    """
    all_scores_list = []
    all_importances_list = []
    all_dimensions_list = []
    all_feature_usage_list = []

    for i in range(n_runs):
        print(f"\n{'='*30}\n         RUN {i+1} / {n_runs}         \n{'='*30}")

        run_scores_df, run_importances_df, run_dimensions_df, run_feature_usage_df = run_single_ml_iteration_weighted(
            df_dict=data_for_analysis_dict,
            target_col=target_col,
            random_seed=i,
            run_number=i + 1,
            use_weights=use_weights,
            encoding=encoding
        )

        all_scores_list.append(run_scores_df)
        all_importances_list.append(run_importances_df)
        all_dimensions_list.append(run_dimensions_df)
        all_feature_usage_list.append(run_feature_usage_df)
        print(f"--- Run {i+1} completed. ---")

    scores_to_concat     = [df for df in all_scores_list      if isinstance(df, pd.DataFrame) and not df.empty]
    importances_to_concat = [df for df in all_importances_list if isinstance(df, pd.DataFrame) and not df.empty]
    dims_to_concat        = [df for df in all_dimensions_list  if isinstance(df, pd.DataFrame) and not df.empty]
    usage_to_concat       = [df for df in all_feature_usage_list if isinstance(df, pd.DataFrame) and not df.empty]

    # No scores from any run means a problem in the input data or the functions above.
    if not scores_to_concat:
        raise RuntimeError("No scores were produced in any run. Check the input data and function definitions.")

    all_scores_df       = pd.concat(scores_to_concat,      ignore_index=True)
    all_importances_df  = pd.concat(importances_to_concat, ignore_index=True) if importances_to_concat else pd.DataFrame()
    all_dimensions_df   = pd.concat(dims_to_concat,        ignore_index=True) if dims_to_concat        else pd.DataFrame()
    all_feature_usage_df = pd.concat(usage_to_concat,      ignore_index=True) if usage_to_concat       else pd.DataFrame()

    # Fails loudly if all_scores_df does not have the expected columns.
    final_summary = all_scores_df.groupby(['ballot', 'year', 'model', 'metric'])['avg_score'].agg(['mean', 'std']).reset_index()
    final_summary = final_summary.rename(columns={'mean': 'final_avg_score', 'std': 'final_std_score'})
    print(f"\n\n{'='*40}\n     FINAL AVERAGED RESULTS ({n_runs} RUNS)     \n{'='*40}")
    print(final_summary[final_summary['metric'] == 'roc_auc_ovr'].round(4))

    return all_scores_df, all_importances_df, final_summary, all_dimensions_df, all_feature_usage_df

## Summary tables

In [ ]:
def compare_model_performance(scores_df, metric='roc_auc_ovr'):
    """Mean/SD of `metric` by (ballot, year, model), sorted."""
    print(f"\n{'='*30}\n   Model performance ({metric})   \n{'='*30}")

    if scores_df.empty or metric not in scores_df['metric'].values:
        print("!! Warning: Scores DataFrame empty/metric not found.")
        return pd.DataFrame(columns=['ballot', 'year', 'model', 'avg_score_mean', 'avg_score_std'])

    metric_df = scores_df[scores_df['metric'] == metric].copy()
    performance_summary = metric_df.groupby(['ballot', 'year', 'model'])['avg_score'].agg(['mean', 'std']).reset_index()
    performance_summary = performance_summary.rename(columns={'mean': 'avg_score_mean', 'std': 'avg_score_std'})
    performance_summary = performance_summary.sort_values(by=['ballot', 'year', 'avg_score_mean'], ascending=[True, True, False])
    print(performance_summary.round(4))
    return performance_summary





def analyze_feature_frequency(importance_df, top_n=5):
    """
    For each (ballot, model, run, year), take the top_n features by importance and count
    how often each feature is in the top N (frequency_top_n) and how often it is used at all (total_occurrences);
    frequency_normalized = frequency_top_n / total_occurrences.
    A high ratio means that whenever the feature is used in that (ballot, model) group,
    it almost always ranks near the top.
    """

    # --- 0. Denominator base ---
    # With the {(ballot, model): {year: df}} structure, 'ballot' and 'year' columns always exist.
    denominator_base_df = importance_df.copy()
    top_n_group_keys = ['run', 'ballot', 'year', 'model']

    # --- 1. Top-N features (numerator base) ---
    # (rows used for 'frequency_top_n')
    top_features = importance_df.sort_values('importance', ascending=False).groupby(top_n_group_keys).head(top_n)

    numerator_base_df = top_features

    # --- 3. Overall frequency (not split by ballot/model) ---
    # Counts only; no normalization because groups are not split.
    overall_analysis = top_features.groupby('feature')['importance'].agg(
        frequency=('count'),
        importance_mean=('mean')
    ).reset_index().sort_values(by='frequency', ascending=False)

    # --- 4. By group: numerator ---
    grouped_analysis_numerator = numerator_base_df.groupby(['ballot', 'model', 'feature'])['importance'].agg(
        frequency_top_n=('count'), # (A) times in the top N
        importance_mean=('mean')
    ).reset_index()

    # --- 5. By group: denominator ---
    grouped_analysis_denominator = denominator_base_df.groupby(['ballot', 'model', 'feature'])['importance'].agg(
        total_occurrences=('count') # (B) times the variable was used
    ).reset_index()

    # --- 6. Merge (A) and (B), normalized frequency ---
    grouped_analysis = pd.merge(
        grouped_analysis_numerator,
        grouped_analysis_denominator,
        on=['ballot', 'model', 'feature'],
        how='left' # keyed on the variables in the top N
    )

    # (A) / (B)
    grouped_analysis['frequency_normalized'] = grouped_analysis['frequency_top_n'].astype(float) / grouped_analysis['total_occurrences']
    grouped_analysis['frequency_normalized'] = grouped_analysis['frequency_normalized'].fillna(0) # guard against NaN

    # --- 7. Sort by normalized frequency ---
    grouped_analysis = grouped_analysis.sort_values(
        by=['ballot', 'model', 'frequency_normalized'],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    # --- 8. Print results by (ballot, model) ---
    print("\n--- Grouped Frequency Analysis (Normalized) ---")
    for name, group in grouped_analysis.groupby(['ballot', 'model']):
        ballot_name, model_name = name
        print(f"\n[Group: {ballot_name} - {model_name}]")
        # Columns to show
        print(group[[
            'feature',
            'frequency_normalized',
            'frequency_top_n',
            'total_occurrences',
            'importance_mean'
        ]].head(5).to_markdown(index=False, floatfmt=".4f"))

    return overall_analysis, grouped_analysis





def analyze_temporal_change(importance_df):
    print(f"\n{'='*40}\n   Feature importance by year (score + rank)   \n{'='*40}")

    if importance_df.empty or 'importance' not in importance_df.columns:
        print("!! Warning: Importance DataFrame empty. Skipping temporal analysis.")
        return {}

    df = importance_df.copy()
    df['importance'] = pd.to_numeric(df['importance'], errors='coerce')
    df = df.dropna(subset=['importance'])

    if df.empty:
        print("!! Warning: No valid numeric importance data for temporal analysis.")
        return {}

    avg_importance_df = df.groupby(['model', 'ballot', 'year', 'feature'])['importance'].mean().reset_index()
    avg_importance_df['rank'] = avg_importance_df.groupby(['model', 'ballot', 'year'])['importance'].rank(method='first', ascending=False).astype(int)

    results_by_model = {}
    for model_name in avg_importance_df['model'].unique():
        model_df = avg_importance_df[avg_importance_df['model'] == model_name]
        for ballot_type in sorted(model_df['ballot'].unique()):
            ballot_df = model_df[model_df['ballot'] == ballot_type]
            # A failing pivot (e.g. duplicated (feature, year) pairs) means importance_df
            # is malformed, so the exception is allowed to stop the analysis.
            pivot_table = ballot_df.pivot_table(index='feature', columns='year', values=['importance', 'rank'])
            pivot_table = pivot_table.swaplevel(0, 1, axis=1).sort_index(axis=1)
            results_by_model[f"Temporal_{model_name}_{ballot_type}"] = pivot_table
    return results_by_model





def analyze_cross_model_change(importance_df):
    print(f"\n{'='*40}\n   Feature importance by model and year   \n{'='*40}")

    if importance_df.empty or 'importance' not in importance_df.columns:
        print("!! Warning: Importance DataFrame empty. Skipping cross-model analysis.")
        return {}

    df = importance_df.copy()
    df['importance'] = pd.to_numeric(df['importance'], errors='coerce')
    df = df.dropna(subset=['importance'])

    if df.empty:
        print("!! Warning: No valid numeric importance data for cross-model analysis.")
        return {}

    avg_importance_df = df.groupby(['model', 'ballot', 'year', 'feature'])['importance'].mean().reset_index()
    avg_importance_df['rank'] = avg_importance_df.groupby(['model', 'ballot', 'year'])['importance'].rank(method='first', ascending=False).astype(int)

    results_by_ballot = {}
    for ballot_type in sorted(avg_importance_df['ballot'].unique()):
        ballot_df = avg_importance_df[avg_importance_df['ballot'] == ballot_type]
        # A failing pivot (e.g. duplicated (feature, year, model) triples) means importance_df
        # is malformed, so the exception is allowed to stop the analysis.
        pivot_table = ballot_df.pivot_table(index='feature', columns=['year', 'model'], values=['importance', 'rank'])
        pivot_table.columns = [f"{year}_{model}_{val}" for val, year, model in pivot_table.columns]
        pivot_table = pivot_table.sort_index(axis=1)
        results_by_ballot[f"CrossModel_{ballot_type}"] = pivot_table
    return results_by_ballot





def save_results_to_excel(results_dict, filename="ml_analysis_summary.xlsx"):
    """Save each item of results_dict as one sheet of an Excel file."""
    with pd.ExcelWriter(filename) as writer:
        for sheet_name, df in results_dict.items():
             if isinstance(df, pd.DataFrame):
                # Ensure sheet names are valid
                safe_sheet_name = str(sheet_name).replace('[','').replace(']','').replace('*','').replace(':','').replace('?','').replace('/','\\')
                safe_sheet_name = safe_sheet_name[:31] # Max length limit

                if isinstance(df.index, (pd.MultiIndex, pd.Index)) and df.index.name is not None:
                    df.to_excel(writer, sheet_name=safe_sheet_name, index=True)
                else:
                    df.to_excel(writer, sheet_name=safe_sheet_name, index=False)
    print(f"\n  All results saved to '{filename}'.")


# --- Full workflow: training -> performance/importance summaries -> Excel/Parquet ---

## Workflow and saving

In [ ]:
import os

# Output folder of this notebook
OUT_DIR = "outputs/1_1_preliminary_logit"

def run_analysis_for_target_weighted_selective(
    df_dict,    # input data: {(ballot, model): {year: df}}
    df_name,    # name used in the output file names (e.g. "Preliminary")
    target_col,
    n_runs,
    encoding='onehot'  # 'onehot' or 'numeric'
):
    """
    Run the full workflow for one target variable and save the results:
    run_ml_multiple_times_weighted -> summary functions -> Excel/Parquet.
    """
    print(f"\n{'='*60}")
    print(f"  Starting Workflow for Target: '{target_col}' ({n_runs} runs)")
    print(f"  Data Source: {df_name}")
    print(f"{'='*60}")

    all_scores, all_importances, final_summary, all_dimension, all_feature_usage = run_ml_multiple_times_weighted(
        data_for_analysis_dict=df_dict,
        target_col=target_col,
        n_runs=n_runs,
        use_weights=True,
        encoding=encoding
    )

    performance_comparison_df = compare_model_performance(all_scores, metric='roc_auc_ovr')
    overall_freq_df, grouped_freq_df = analyze_feature_frequency(all_importances, top_n=20)
    temporal_results    = analyze_temporal_change(all_importances)
    cross_model_results = analyze_cross_model_change(all_importances)

    print(f"\n--- [{target_col} - {df_name}] Collecting results... ---")

    # Fails loudly if all_dimension does not have the expected columns.
    avg_dims = all_dimension.groupby(['ballot', 'year', 'model'])[['n_rows', 'n_cols']].mean().reset_index()

    # Feature usage log of run 1 (the feature set is identical across runs)
    unique_feature_usage_df = (
        all_feature_usage[all_feature_usage['run'] == 1]
        .drop(columns=['run'])
        .reset_index(drop=True)
    )

    excel_sheets = {
        "0_Final_Summary":      final_summary,
        "1_Dataset_Dimensions": avg_dims,
        "2_Performance_Compare": performance_comparison_df,
        "3_Feature_Freq_Overall": overall_freq_df,
        "4_Feature_Freq_Grouped": grouped_freq_df,
        "5_Feature_Usage_Log":   unique_feature_usage_df,
        **temporal_results,
        **cross_model_results
    }

    os.makedirs(OUT_DIR, exist_ok=True)
    excel_filename   = os.path.join(OUT_DIR, f"1_{df_name}_{target_col}_analysis_results_weighted.xlsx")
    import_filename  = os.path.join(OUT_DIR, f"1_{df_name}_{target_col}_analysis_all_importance_weighted.parquet")
    scores_filename  = os.path.join(OUT_DIR, f"1_{df_name}_{target_col}_analysis_all_scores_weighted.parquet")
    usage_filename   = os.path.join(OUT_DIR, f"1_{df_name}_{target_col}_analysis_feature_usage_log.parquet")

    save_results_to_excel(excel_sheets, filename=excel_filename)

    print(f"\n--- [{target_col} - {df_name}] Saving parquet files... ---")
    if not all_importances.empty: all_importances.to_parquet(import_filename)
    if not all_scores.empty:      all_scores.to_parquet(scores_filename)
    if not all_feature_usage.empty: all_feature_usage.to_parquet(usage_filename)

    print(f"  [{target_col} - {df_name}] All files saved.")
    print(f"{'-'*60}\n")

## Run

In [ ]:
import pickle
import os

# Same preliminary input as 02_preliminary.ipynb (columns with >5% missing dropped + list-wise deletion in 01_make_data.ipynb)
filename = "data/cleaned_data_for_preliminary_analysis.pkl"

# Stop if the input file is missing.
if not os.path.exists(filename):
    raise FileNotFoundError(f"'{filename}' not found. Run 01_make_data.ipynb first.")

with open(filename, 'rb') as file:
    data_loaded = pickle.load(file)

# The pkl stores each ballot's data under the XGBoost key; the same data are used
# under the 'Logit' key (identical to the XGBoost input of 02_preliminary.ipynb).
data_for_analysis = {(ballot, 'Logit'): year_dict
                     for (ballot, model_name), year_dict in data_loaded.items() if model_name == 'XGBoost'}
if sorted(data_for_analysis.keys()) != [('a', 'Logit'), ('b', 'Logit'), ('c', 'Logit')]:
    raise ValueError(f"Unexpected keys: {sorted(data_for_analysis.keys())}")
print(f"Loaded '{filename}': {list(data_for_analysis.keys())}, "
      f"{sum(len(v) for v in data_for_analysis.values())} ballot-model-year cells")

In [ ]:
N_ITERATIONS_REDUCED = 20
TARGET_VARIABLES = ['partyid_1']
# Result file name (df_name) by encoding: 'onehot' = main version, 'numeric' = same numeric input as XGBoost
ENCODINGS = {'Logit_OneHot': 'onehot', 'Logit_Numeric': 'numeric'}

for target in TARGET_VARIABLES:
    for df_name, encoding in ENCODINGS.items():
        run_analysis_for_target_weighted_selective(
            df_dict=data_for_analysis,
            df_name=df_name,
            target_col=target,
            n_runs=N_ITERATIONS_REDUCED,
            encoding=encoding
        )